# KPI Data Checks for Retail Sales ELT
Notebook ini memverifikasi kondisi data yang mendukung KPI dashboard: Total Sales, Total Profit, Profit Margin, Monthly Sales Trend, Top Products, dan Sales per Region.

## 1. Setup dan koneksi

In [ ]:
import os
import pandas as pd
import clickhouse_connect
from dotenv import load_dotenv

load_dotenv(dotenv_path='../.env')
RAW_DIR = os.getenv('RAW_DIR', 'data')
CH_HOST = os.getenv('CH_HOST', 'localhost')
CH_PORT = int(os.getenv('CH_PORT', 8123))
CH_USER = os.getenv('CH_USER', 'admin')
CH_PASSWORD = os.getenv('CH_PASSWORD', 'admin123')

client = clickhouse_connect.get_client(
    host=CH_HOST,
    port=CH_PORT,
    username=CH_USER,
    password=CH_PASSWORD,
)
print('✅ Connected to ClickHouse')

## 2. Bronze ingestion validation

In [ ]:
csv_path = os.path.join('..', RAW_DIR, 'samplesuperstore.csv')
df_csv = pd.read_csv(csv_path, encoding='utf-8')
csv_count = len(df_csv)

table_name = 'bronze.bronze_orders'
result = client.query(f'SELECT count() FROM {table_name}')
ch_count = result.result_rows[0][0]
print(f'CSV rows          : {csv_count:,}')
print(f'Bronze rows       : {ch_count:,}')
print('✅ Bronze ingestion row count check' if ch_count == csv_count else '⚠️ Bronze ingestion mismatch')

## 3. Silver model sanity checks

In [ ]:
silver_table = 'silver.silver_orders'
df_silver = client.query_df(f'SELECT * FROM {silver_table} LIMIT 5')
print('Silver preview:')
display(df_silver)

schema_silver = client.query_df(
    f"SELECT name, type FROM system.columns WHERE database='silver' AND table='silver_orders' ORDER BY position"
)
print('Silver schema:')
display(schema_silver)

## 4. KPI checks for dashboard metrics

In [ ]:
print('### KPI Card values')
kpi_query = '''
SELECT
    sum(sales) AS total_sales,
    sum(profit) AS total_profit,
    round(100.0 * sum(profit) / NULLIF(sum(sales), 0), 2) AS profit_margin
FROM silver.silver_orders
'''
df_kpi = client.query_df(kpi_query)
display(df_kpi)

### Monthly Sales Trend

In [ ]:
trend_query = '''
SELECT
    toStartOfMonth(order_date) AS order_month,
    sum(sales) AS total_sales,
    sum(profit) AS total_profit,
    round(100.0 * sum(profit) / NULLIF(sum(sales), 0), 2) AS profit_margin
FROM silver.silver_orders
GROUP BY order_month
ORDER BY order_month
'''
df_trend = client.query_df(trend_query)
display(df_trend)

### Top Products by Profit and Sales
Top 5 produk akan terlihat dari tabel ringkasan produk berikut.

In [ ]:
product_query = '''
SELECT
    product_id,
    product_name,
    category,
    sub_category,
    total_sales,
    total_profit,
    profit_margin,
    total_quantity
FROM gold.product_performance
ORDER BY total_profit DESC
LIMIT 5
'''
df_top_profit = client.query_df(product_query)
print('Top 5 products by profit:')
display(df_top_profit)

product_query_sales = '''
SELECT
    product_id,
    product_name,
    category,
    sub_category,
    total_sales,
    total_profit,
    profit_margin,
    total_quantity
FROM gold.product_performance
ORDER BY total_sales DESC
LIMIT 5
'''
df_top_sales = client.query_df(product_query_sales)
print('Top 5 products by sales:')
display(df_top_sales)

### Sales per Region

In [ ]:
region_query = '''
SELECT
    region,
    total_sales,
    total_profit,
    profit_margin,
    total_quantity
FROM gold.region_sales
ORDER BY total_sales DESC
'''
df_region = client.query_df(region_query)
display(df_region)

## 5. Data condition checks untuk KPI

In [ ]:
checks = {
    'null_order_id': 'SELECT count() FROM silver.silver_orders WHERE order_id IS NULL',
    'null_order_date': 'SELECT count() FROM silver.silver_orders WHERE order_date IS NULL',
    'null_sales': 'SELECT count() FROM silver.silver_orders WHERE sales IS NULL',
    'negative_sales': 'SELECT count() FROM silver.silver_orders WHERE sales < 0',
    'negative_profit': 'SELECT count() FROM silver.silver_orders WHERE profit < 0',
    'duplicate_order_ids': 'SELECT count() - count(distinct order_id) FROM silver.silver_orders',
    'order_after_ship': 'SELECT count() FROM silver.silver_orders WHERE order_date > ship_date',
    'zero_quantity': 'SELECT count() FROM silver.silver_orders WHERE quantity <= 0',
}
for name, query in checks.items():
    result = client.query_df(query)
    print(f'{name}:', int(result.iloc[0, 0]))